<a href="https://colab.research.google.com/github/msfasha/307401-Big-Data/blob/main/20252/Module%205%20-%20Introduction%20to%20Spark/Spark_Essentials.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

<h1 style="color:#E25822;font-family:Arial;font-size:2.4em;border-bottom:3px solid #E25822;padding-bottom:10px;">
  Apache Spark — Theory and Practice
</h1>

**Course:** Big Data Analytics &nbsp;|&nbsp; **Module 5:** Introduction to Apache Spark  
**Prerequisites:** Python, SQL, basic understanding of distributed systems

---

## Module Learning Objectives

By completing this module, students will be able to:

1. Explain Spark's cluster architecture and execution model (Jobs → Stages → Tasks)
2. Describe how Catalyst and Tungsten optimize query execution
3. Distinguish between narrow and wide transformations and predict their cost
4. Build streaming pipelines with window operations and watermarking
5. Design end-to-end ML Pipelines using Spark MLlib
6. Diagnose and fix performance bottlenecks using caching, partitioning, and broadcast joins

---

## Module Contents

| Section | Topics | Type |
|---------|--------|------|
| 1 — Architecture & Internals | Cluster model, DAG, Catalyst, Tungsten, fault tolerance | Theory + Lab |
| 2 — DataFrames & Spark SQL | Schema, transformations, actions, window functions | Lab |
| 3 — Structured Streaming | Micro-batch, windows, watermarking, ETL pattern | Theory + Lab |
| 4 — Machine Learning (MLlib) | Pipelines, classification, regression, clustering, tuning | Lab |
| 5 — Performance Optimization | Caching, partitioning, broadcast joins, AQE | Theory + Lab |

In [ ]:
!pip install pyspark --quiet

In [ ]:
import time, os, shutil, random, math, csv, glob, tempfile
from datetime import datetime, timedelta

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.functions import broadcast
from pyspark.storagelevel import StorageLevel

spark = (
    SparkSession.builder
    .appName("BigData-Module5")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.adaptive.enabled", "true")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
TMP = tempfile.mkdtemp()

print(f"Spark {spark.version} | Cores: {spark.sparkContext.defaultParallelism}")
print(f"Temp dir: {TMP}")

---
<h2 style="color:#E25822;font-family:Arial;">Section 1 — Spark Architecture & Internals</h2>

### Learning Objectives
- Describe the roles of Driver, Executor, and Cluster Manager
- Trace the execution path from a user action to individual tasks
- Explain how Catalyst's four optimization phases improve performance
- Explain how Tungsten's memory management differs from standard JVM execution
- Describe Spark's lineage-based fault tolerance model

### 1.1 Cluster Architecture

A Spark application runs on a **cluster** consisting of three logical components:

```
┌────────────────────────────────────────────────────────────────────┐
│                        SPARK APPLICATION                           │
│                                                                    │
│  ┌──────────────────┐         ┌────────────────────────────────┐  │
│  │   DRIVER PROGRAM │         │       CLUSTER MANAGER          │  │
│  │                  │◀───────▶│  (YARN / Kubernetes /          │  │
│  │  - SparkContext  │  request│   Mesos / Standalone)          │  │
│  │  - SparkSession  │  workers│                                │  │
│  │  - DAG Scheduler │         └────────────────────────────────┘  │
│  │  - Task Scheduler│                        │ allocate            │
│  └──────────┬───────┘                        ▼                    │
│             │ dispatch tasks     ┌────────────────────────┐        │
│             │                   │      Worker Node 1      │        │
│             ├──────────────────▶│  ┌─────────────────┐   │        │
│             │                   │  │   EXECUTOR JVM  │   │        │
│             │                   │  │  - Thread pool  │   │        │
│             │                   │  │  - Task Task    │   │        │
│             │                   │  │  - Block Manager│   │        │
│             │                   │  └─────────────────┘   │        │
│             │                   └────────────────────────┘        │
│             │                   ┌────────────────────────┐        │
│             └──────────────────▶│      Worker Node 2      │        │
│                                 │  ┌─────────────────┐   │        │
│                                 │  │   EXECUTOR JVM  │   │        │
│                                 │  │  - Thread pool  │   │        │
│                                 │  │  - Task Task    │   │        │
│                                 │  └─────────────────┘   │        │
│                                 └────────────────────────┘        │
└────────────────────────────────────────────────────────────────────┘
```

**Driver** — the JVM process running your application's `main()` function. It:
- Constructs the logical plan (DAG) from your transformations
- Negotiates resources with the Cluster Manager
- Schedules tasks onto Executors
- Collects results from `collect()` and `show()`

**Executor** — a long-lived JVM process on each worker node. It:
- Receives and executes tasks dispatched by the Driver
- Stores cached (persisted) partition data in its Block Manager
- Runs multiple tasks concurrently (one per CPU core by default)

**Cluster Manager** — external resource arbitrator. Spark supports four:
- **Standalone** — Spark's built-in scheduler (simplest setup)
- **YARN** — Hadoop's resource manager (most common in enterprise)
- **Kubernetes** — container-orchestrated deployment (growing adoption)
- **Mesos** — fine-grained resource sharing (declining use)

> **Important:** On Google Colab and in local development you run in `local[*]` mode — the Driver and a single Executor share the same JVM process on one machine. All concepts still apply; you are simulating a cluster on one node.

### 1.2 SparkSession vs SparkContext

A frequent source of confusion — especially when reading Stack Overflow answers or older textbooks.

| | **SparkContext** | **SparkSession** |
|--|-----------------|-----------------|
| Introduced | Spark 1.0 | Spark 2.0 |
| Entry point for | RDDs only | DataFrames, SQL, Hive, Streaming, ML |
| Creation | `sc = SparkContext(conf)` | `spark = SparkSession.builder...getOrCreate()` |
| Relationship | Exists standalone | **Wraps SparkContext internally** |
| Status today | Legacy; still present | **The correct API to use** |

When you call `SparkSession.builder.getOrCreate()`, Spark automatically creates an underlying `SparkContext`.
You can access it via `spark.sparkContext`, but you should only need it for low-level RDD operations — which in production code are rare.

> **RDD note:** Spark's original abstraction, the **Resilient Distributed Dataset (RDD)**, is a typed, immutable,
> partitioned collection processed via functional transformations (map, filter, reduce).
> DataFrames are built *on top of* RDDs but expose a higher-level, schema-aware API that Catalyst can optimize.
> Write DataFrames. Understand that RDDs exist underneath.

In [ ]:
# SparkContext is accessible through SparkSession — rarely needed directly
sc = spark.sparkContext

print(f"SparkSession version   : {spark.version}")
print(f"SparkContext app name  : {sc.appName}")
print(f"SparkContext master    : {sc.master}")
print(f"SparkContext UI        : {sc.uiWebUrl}")
print(f"Same object?           : {spark.sparkContext is sc}")
print(f"Default parallelism    : {sc.defaultParallelism}  (= number of CPU cores in local mode)")

### 1.3 The Execution Model: Jobs → Stages → Tasks

When you call an **action** (any operation that returns a result or writes data), Spark executes the following pipeline:

```
User calls an action (e.g., count(), show(), write())
              │
              ▼
     DAG Scheduler analyses the computation graph
     and partitions it into STAGES at shuffle boundaries
              │
    ┌─────────┴────────────┐
    │                      │
    ▼                      ▼
 STAGE 1               STAGE 2
 (before shuffle)      (after shuffle)
 ┌──┐ ┌──┐ ┌──┐        ┌──┐ ┌──┐ ┌──┐
 │T1│ │T2│ │T3│  ─────▶ │T4│ │T5│ │T6│
 └──┘ └──┘ └──┘  shuffle└──┘ └──┘ └──┘
  one task per            one task per
  input partition         output partition
```

**Stage boundaries** are determined by **shuffle dependencies** — transformations that require
data to be redistributed across partitions (and therefore across the network).

#### Narrow vs Wide Transformations

| Type | Definition | Examples | Stage boundary? |
|------|-----------|---------|-----------------|
| **Narrow** | Each output partition depends on exactly one input partition | `filter`, `select`, `map`, `withColumn`, `union` | No |
| **Wide** | Each output partition depends on multiple input partitions (requires shuffle) | `groupBy`, `join`, `distinct`, `orderBy`, `repartition` | **Yes** |

**Why shuffles are expensive:** During a shuffle, each task:
1. **Writes** its output records to local disk, sorted by destination partition
2. Records are **transferred over the network** to the appropriate executor
3. Receiving tasks **read and merge** the data before processing

This involves disk I/O + network I/O + sorting — typically the dominant cost in a Spark job.

#### Lazy Evaluation

Spark applies **lazy evaluation**: transformations are not executed when called.
Instead, Spark records each transformation in the **lineage graph** (DAG) and executes the
full graph only when an action is triggered. Benefits:

- **Pipeline optimization:** Catalyst can reorder, merge, and eliminate steps before running
- **Fault recovery:** If a partition is lost, Spark re-executes only the lineage needed to rebuild it
- **Efficiency:** A call to `first()` stops execution as soon as one record is found

In [ ]:
# Demonstrating lazy evaluation: transformations are recorded, not executed

df_lazy = spark.range(5_000_000).withColumn("value", F.rand(seed=42) * 1000)

t0 = time.time()
# Four chained transformations — all lazy
step1 = df_lazy.filter(F.col("value") > 500)
step2 = step1.withColumn("category", (F.col("id") % 10).cast(StringType()))
step3 = step2.filter(F.col("category") == "3")
step4 = step3.select("id", "value", "category")
t_transform = time.time() - t0

print(f"4 transformations recorded in {t_transform*1000:.1f} ms  (nothing ran yet)")
print(f"Is streaming: {step4.isStreaming}")

# Action triggers execution of the entire DAG
t0 = time.time()
result = step4.count()
t_action = time.time() - t0

print(f"Action .count() completed in {t_action:.3f}s  (full DAG executed here)")
print(f"Result: {result:,} rows")

# Inspect the optimized physical plan Catalyst produced
print("\n── Physical Execution Plan (read bottom-up) ──")
step4.explain()

### 1.4 The Catalyst Optimizer

**Catalyst** is Spark SQL's extensible query optimizer. It processes every DataFrame operation
through four sequential phases before generating executable code:

```
User Query (Python/SQL)
        │
        ▼
┌───────────────────────────────────────────────────┐
│  Phase 1: ANALYSIS                                │
│  • Resolve column names against the catalog       │
│  • Infer and validate data types                  │
│  • Detect ambiguous references                    │
└─────────────────────┬─────────────────────────────┘
                      │
                      ▼
┌───────────────────────────────────────────────────┐
│  Phase 2: LOGICAL OPTIMIZATION                    │
│  • Predicate pushdown — move filters toward source│
│  • Column pruning — drop unreferenced columns     │
│  • Constant folding — evaluate 1+2 → 3 at plan time│
│  • Boolean simplification                        │
│  • Join reordering                               │
└─────────────────────┬─────────────────────────────┘
                      │
                      ▼
┌───────────────────────────────────────────────────┐
│  Phase 3: PHYSICAL PLANNING                       │
│  • Select join strategy (broadcast / sort-merge)  │
│  • Choose sort algorithm                          │
│  • Estimate cost using statistics                 │
│  • Generate one or more candidate plans, pick best│
└─────────────────────┬─────────────────────────────┘
                      │
                      ▼
┌───────────────────────────────────────────────────┐
│  Phase 4: CODE GENERATION (Tungsten)              │
│  • Compile plan to JVM bytecode                   │
│  • Whole-stage code generation                    │
│  • Vectorized execution where possible            │
└─────────────────────┬─────────────────────────────┘
                      │
                      ▼
              Executed on Cluster
```

**Why Catalyst matters for DataFrames vs RDDs:**
Catalyst can only optimize operations it understands structurally.
A DataFrame operation (`F.col("salary") * 1.1`) is expressed as a *logical expression tree*
that Catalyst can manipulate. An RDD lambda (`lambda x: x['salary'] * 1.1`) is an opaque
Python function — Catalyst cannot look inside it, reorder it, or optimize it.

In [ ]:
# Write a Parquet dataset to demonstrate Catalyst optimizations
parquet_path = os.path.join(TMP, "employees")

(spark.range(500_000)
 .withColumn("department", F.element_at(
     F.array(*[F.lit(d) for d in ["Engineering","Marketing","Finance","HR","Sales"]]),
     (F.col("id") % 5 + 1).cast(IntegerType())))
 .withColumn("salary",     F.rand(seed=1) * 90000 + 30000)
 .withColumn("age",        (F.rand(seed=2) * 40 + 22).cast(IntegerType()))
 .withColumn("years_exp",  (F.rand(seed=3) * 20 + 1).cast(IntegerType()))
 .withColumn("country",    F.lit("Jordan"))
 .write.mode("overwrite").parquet(parquet_path)
)

df_emp = spark.read.parquet(parquet_path)
print("Dataset written. Schema:")
df_emp.printSchema()

In [ ]:
# ── Predicate Pushdown ─────────────────────────────────────────────────────
# Catalyst moves the filter INTO the Parquet reader so non-matching rows
# are never even deserialized. Look for "PushedFilters" in the plan.

q_pushdown = (df_emp
    .filter(F.col("department") == "Engineering")
    .filter(F.col("salary") > 70000)
    .select("id", "salary", "years_exp"))

print("=" * 65)
print("Query: filter(dept=Engineering, salary>70k) → select(id, salary, years_exp)")
print("=" * 65)
q_pushdown.explain()
print("▶ PushedFilters: both predicates are pushed into the Parquet Scan")
print("▶ ReadSchema: only id, salary, years_exp are read — 'age', 'country' never loaded")

In [ ]:
# ── Constant Folding and Plan Fusion ───────────────────────────────────────
# Catalyst evaluates constant expressions at plan-time and fuses filter steps

q_fusion = (df_emp
    .filter(F.col("age") > 20 + 20)         # constant: 20+20 = 40
    .filter(F.col("salary") > 50000)
    .filter(F.col("years_exp") >= 5)
    .select("id", "department", "salary"))

print("=" * 65)
print("Three separate filters — does Catalyst merge them?")
print("=" * 65)
q_fusion.explain()
print("▶ A single Filter node in the plan indicates Catalyst fused all three")
print("▶ The constant 20+20 is folded to 40 at plan time, not at runtime")

### 1.5 Tungsten — The Execution Engine

While Catalyst decides *what* operations to run, **Tungsten** (introduced in Spark 1.4,
significantly enhanced in 2.0) decides *how* to execute them efficiently at the hardware level.

#### Three Core Mechanisms

**1. Off-Heap Memory Management**

Standard JVM programs store objects on the heap, managed by the Garbage Collector (GC).
The GC periodically pauses all threads to reclaim memory — in large Spark jobs these pauses
can last seconds, causing task failures and severe performance degradation.

Tungsten bypasses the GC by managing memory directly using `sun.misc.Unsafe`, an internal
Java API that allows raw memory allocation. Data is stored in **compact binary format** 
(similar to a C struct) rather than as Java objects, reducing memory footprint by 2–5×.

**2. Cache-Aware Computation**

Modern CPUs execute fastest when data fits in L1/L2/L3 cache. Tungsten layouts data contiguously
in memory, maximizing cache line utilization during operations like sorting and hashing.

**3. Whole-Stage Code Generation**

The traditional interpreter model processes one row at a time through a chain of virtual
function calls — each call has overhead. Tungsten compiles the entire query stage into a
**single tight JVM bytecode loop**, eliminating virtual dispatch and enabling JIT compilation
to optimize the hot loop. This is called *whole-stage code generation*.

```
Traditional (interpreted):          Tungsten (code-generated):
for each row:                       compiled single loop:
  apply Filter                        for each row in partition:
  for each row:                         if (salary > 70000 &&
    apply Project                           dept == "Engineering")
    for each row:                         output(id, salary, years_exp)
      apply Aggregate                 // JIT-compiled to native machine code
```

#### Observable consequence: avoid Python UDFs

A Python UDF forces Spark to:
1. Serialize each row from JVM binary format → Python pickle format
2. Send the row across a local socket to a Python process
3. Execute the Python function
4. Serialize the result back → JVM binary format

This completely bypasses Tungsten's code generation and typically costs **5–20× more** than an equivalent built-in expression.

In [ ]:
from pyspark.sql.functions import udf
import pandas as pd

df_udf = spark.range(1_000_000).withColumn("salary", F.rand(seed=5) * 90000 + 30000)

# Method 1: Built-in expression — Tungsten-optimized, stays in JVM
t0 = time.time()
df_udf.withColumn("band",
    F.when(F.col("salary") < 50000, "Low")
     .when(F.col("salary") < 80000, "Mid")
     .otherwise("High")).count()
t_builtin = time.time() - t0

# Method 2: Python UDF — breaks Tungsten, serializes every row to Python
@udf(StringType())
def band_udf(salary):
    if salary < 50000: return "Low"
    elif salary < 80000: return "Mid"
    else: return "High"

t0 = time.time()
df_udf.withColumn("band", band_udf(F.col("salary"))).count()
t_py_udf = time.time() - t0

# Method 3: Pandas UDF (vectorized) — Arrow-based batch transfer, much better
try:
    from pyspark.sql.functions import pandas_udf

    @pandas_udf(StringType())
    def band_pandas_udf(s: pd.Series) -> pd.Series:
        return s.apply(lambda x: "Low" if x < 50000 else ("Mid" if x < 80000 else "High"))

    t0 = time.time()
    df_udf.withColumn("band", band_pandas_udf(F.col("salary"))).count()
    t_pandas_udf = time.time() - t0
    has_pandas = True
except Exception:
    has_pandas = False

print(f"{'Method':<30} {'Time':>8} {'vs Built-in':>12}")
print("─" * 52)
print(f"{'Built-in F.when() [Tungsten]':<30} {t_builtin:>7.2f}s {'1.0×':>12}")
if has_pandas:
    print(f"{'Pandas UDF [vectorized]':<30} {t_pandas_udf:>7.2f}s {f'{t_pandas_udf/t_builtin:.1f}×':>12}")
print(f"{'Python UDF [row-by-row]':<30} {t_py_udf:>7.2f}s {f'{t_py_udf/t_builtin:.1f}×':>12}")
print("\n→ Use F.* built-ins whenever possible. Use pandas_udf if a UDF is unavoidable.")

### 1.6 Fault Tolerance — Lineage-Based Recovery

Spark achieves fault tolerance without replication through **RDD lineage** (also called the DAG lineage).

Every RDD or DataFrame partition records the *sequence of operations* (its lineage) needed to
recompute it from the original stable storage. If an executor fails and a partition is lost,
the Driver simply re-schedules the tasks needed to recompute that partition from its last stable
ancestor — no data needs to be replicated for this to work.

```
Source (HDFS/S3)
    │
    ▼ read partition p2
Filter(salary > 50000)                ← these steps form the lineage of partition p2
    │
    ▼
GroupBy(department)
    │
    ▼
Partition p2  ← LOST (executor crashed)

Recovery: Spark re-reads source partition 2, re-applies Filter and GroupBy.
No data was replicated. The lineage graph is the recovery mechanism.
```

#### When lineage is insufficient: Checkpointing

For **iterative algorithms** (e.g., PageRank, gradient descent) the lineage grows with each
iteration — eventually the lineage itself becomes too expensive to store and re-execute.
**Checkpointing** materializes the DataFrame to stable storage (HDFS/S3), truncating the lineage.

```python
spark.sparkContext.setCheckpointDir("hdfs:///checkpoints/")
df.checkpoint()   # materializes current state; lineage reset to checkpoint
```

Structured Streaming also uses checkpointing to provide **exactly-once semantics**:
the checkpoint stores query progress and unprocessed offsets so a restarted stream
can resume exactly where it stopped.

In [ ]:
# Demonstrating lineage: show the execution plan for a multi-step pipeline
df_lineage = (
    spark.read.parquet(parquet_path)
    .filter(F.col("department").isin(["Engineering", "Finance"]))
    .withColumn("senior", F.col("years_exp") >= 10)
    .groupBy("department", "senior")
    .agg(F.avg("salary").alias("avg_salary"), F.count("*").alias("headcount"))
    .orderBy("department", "senior")
)

print("Full optimized plan for a 4-step pipeline:")
df_lineage.explain(mode="extended")   # shows all 4 Catalyst phases

### Section 1 — Exercises

**Exercise 1.1 — Execution Model Analysis**

Given the following PySpark code:

```python
df = spark.read.parquet("employees.parquet")        # (A)
filtered = df.filter(F.col("salary") > 60000)       # (B)
grouped  = filtered.groupBy("department")            # (C)
result   = grouped.agg(F.avg("salary"))              # (D)
result.show()                                        # (E)
```

Answer the following:

1. At which line(s) does Spark actually execute computation? Justify your answer using the concept of lazy evaluation.
2. Between lines (C) and (D), which transformation introduces a stage boundary and why? What network operations does this trigger?
3. If an executor fails while processing line (E) and loses a result partition, how does Spark recover? What are the limits of this approach?
4. Rewrite this query using Spark SQL (`spark.sql(...)`) and explain why Catalyst produces an identical physical plan for both versions.

---

**Exercise 1.2 — Catalyst Optimization**

Run the two queries below and call `.explain()` on each. 

```python
# Query A
df.select("id","salary","department").filter(F.col("salary") > 70000)

# Query B  
df.filter(F.col("salary") > 70000).select("id","salary","department")
```

1. Do both queries produce the same physical plan? Run them and compare the output of `.explain()`. Explain your finding.
2. What does this tell you about Catalyst's predicate pushdown capability?
3. Now add `.filter(F.col("department") == "Engineering")` before the `select()` in Query A. Does the plan change? Why?

In [ ]:
# ── Exercise 1.2 workspace ───────────────────────────────────────────────────
df_ex = spark.read.parquet(parquet_path)

# Query A: select then filter
query_A = df_ex.select("id","salary","department").filter(F.col("salary") > 70000)

# Query B: filter then select
query_B = df_ex.filter(F.col("salary") > 70000).select("id","salary","department")

print("=== Query A plan ===")
query_A.explain()

print("=== Query B plan ===")
query_B.explain()

# TODO: Add .filter(department == Engineering) to Query A and compare
# TODO: Write your analysis in a markdown cell below

**Exercise 1.3 — UDF Cost Analysis**

1. Using the UDF benchmark above, calculate the **per-row overhead** of the Python UDF
   compared to the built-in `F.when()` on your machine.
   Show your calculation: `overhead_per_row = (t_udf − t_builtin) / n_rows`

2. Assume a production dataset of **500 million rows**. Extrapolate the estimated wall-clock
   time difference between using `F.when()` and a Python UDF for this dataset size.
   State any assumptions you make.

3. A teammate argues: *"The Python UDF is more readable. The performance difference only
   matters at huge scale. Our dataset is only 50 GB so it's fine."*
   Construct a technical counter-argument using what you know about Tungsten and serialization overhead.

4. Under what circumstances would you still choose a Python UDF over a built-in expression?
   Give one concrete example.

In [ ]:
# ── Exercise 1.3 workspace ───────────────────────────────────────────────────
N_ROWS = 1_000_000

# Overhead per row (fill in from your benchmark above)
# overhead_per_row = (t_py_udf - t_builtin) / N_ROWS
# print(f"Overhead per row: {overhead_per_row*1e6:.2f} microseconds")

# Extrapolation to 500M rows
# LARGE_N = 500_000_000
# estimated_udf_time   = ...
# estimated_builtin_time = ...
# print(...)

---
<h2 style="color:#E25822;font-family:Arial;">Section 2 — DataFrames & Spark SQL</h2>

### Learning Objectives
- Create DataFrames from multiple sources (in-memory, CSV, Parquet, JSON)
- Apply and combine transformations and actions correctly
- Use Spark SQL and understand its equivalence to the DataFrame API
- Apply window functions for ranking, running totals, and time-series analysis

### 2.1 The DataFrame Abstraction

A **DataFrame** is a distributed, immutable collection of rows organized into named, typed columns.
It is conceptually equivalent to a relational table or a pandas DataFrame, but physically
distributed across partitions on a cluster.

```
DataFrame (500,000 rows, 6 columns)
┌──────────┬────────────────┬──────────┬─────┬────────────┬────────────┐
│ order_id │ customer_id    │ product  │ qty │ unit_price │ order_date │
├──────────┼────────────────┼──────────┼─────┼────────────┼────────────┤
│ ...      │ ...            │ ...      │ ... │ ...        │ ...        │  Partition 0
├──────────┼────────────────┼──────────┼─────┼────────────┼────────────┤
│ ...      │ ...            │ ...      │ ... │ ...        │ ...        │  Partition 1
├──────────┼────────────────┼──────────┼─────┼────────────┼────────────┤
│ ...      │ ...            │ ...      │ ... │ ...        │ ...        │  Partition 2
└──────────┴────────────────┴──────────┴─────┴────────────┴────────────┘
```

**Transformations** (lazy — return a new DataFrame):
`filter`, `select`, `withColumn`, `drop`, `groupBy`, `agg`, `join`, `union`,
`orderBy`, `distinct`, `repartition`, `coalesce`

**Actions** (eager — trigger execution, return a result):
`count`, `show`, `collect`, `first`, `take`, `write`, `foreach`, `toPandas`

In [ ]:
# ── Realistic e-commerce dataset ────────────────────────────────────────────
random.seed(42)

CITIES      = ["Amman","Irbid","Zarqa","Aqaba","Madaba","Jerash","Karak"]
CATEGORIES  = ["Electronics","Clothing","Food & Beverage","Sports","Books","Home & Garden"]
PRODUCTS    = {
    "Electronics":      ["Laptop","Smartphone","Tablet","Headphones","Camera"],
    "Clothing":         ["T-Shirt","Jeans","Jacket","Shoes","Hat"],
    "Food & Beverage":  ["Coffee","Tea","Juice","Snacks","Chocolate"],
    "Sports":           ["Football","Basketball","Yoga Mat","Running Shoes","Bicycle"],
    "Books":            ["Novel","Textbook","Comic","Journal","Cookbook"],
    "Home & Garden":    ["Lamp","Chair","Plant Pot","Blanket","Candle"],
}

def random_date(start="2024-01-01", days=365):
    base = datetime(2024,1,1)
    return (base + timedelta(days=random.randint(0, days))).strftime("%Y-%m-%d")

rows = []
for order_id in range(1, 50_001):
    cat      = random.choice(CATEGORIES)
    product  = random.choice(PRODUCTS[cat])
    city     = random.choice(CITIES)
    qty      = random.randint(1, 10)
    price    = round(random.uniform(2.5, 500.0), 2)
    cust_id  = random.randint(1, 5000)
    rows.append((order_id, cust_id, cat, product, city, qty, price, random_date()))

orders_schema = StructType([
    StructField("order_id",    IntegerType(), False),
    StructField("customer_id", IntegerType(), False),
    StructField("category",    StringType(),  False),
    StructField("product",     StringType(),  False),
    StructField("city",        StringType(),  False),
    StructField("quantity",    IntegerType(), False),
    StructField("unit_price",  DoubleType(),  False),
    StructField("order_date",  StringType(),  False),
])

orders = spark.createDataFrame(rows, schema=orders_schema)
orders = orders.withColumn("order_date", F.to_date("order_date","yyyy-MM-dd"))
orders = orders.withColumn("revenue", F.round(F.col("quantity") * F.col("unit_price"), 2))

orders.write.mode("overwrite").parquet(os.path.join(TMP,"orders.parquet"))
orders = spark.read.parquet(os.path.join(TMP,"orders.parquet"))

print(f"Orders dataset: {orders.count():,} rows")
orders.printSchema()
orders.show(5)

In [ ]:
# ── Core DataFrame Operations ────────────────────────────────────────────────

# 1. Aggregation: revenue by category
print("─ Revenue by Category ─")
orders.groupBy("category")     .agg(
        F.round(F.sum("revenue"),    2).alias("total_revenue"),
        F.round(F.avg("unit_price"), 2).alias("avg_price"),
        F.countDistinct("product")   .alias("unique_products"),
        F.count("*")                 .alias("order_count")
    )     .orderBy(F.desc("total_revenue"))     .show()

# 2. Filtering and derived columns
print("─ High-Value Orders (revenue > 1000 JOD) ─")
high_value = orders     .filter(F.col("revenue") > 1000)     .withColumn("revenue_tier",
        F.when(F.col("revenue") > 3000, "Platinum")
         .when(F.col("revenue") > 2000, "Gold")
         .otherwise("Silver"))     .select("order_id","customer_id","product","revenue","revenue_tier","city")

high_value.groupBy("revenue_tier").count().orderBy("revenue_tier").show()

In [ ]:
# ── Spark SQL — identical results, same Catalyst plan ──────────────────────
orders.createOrReplaceTempView("orders")

sql_result = spark.sql("""
    SELECT
        category,
        ROUND(SUM(revenue),    2) AS total_revenue,
        ROUND(AVG(unit_price), 2) AS avg_price,
        COUNT(DISTINCT product)   AS unique_products,
        COUNT(*)                  AS order_count
    FROM orders
    GROUP BY category
    ORDER BY total_revenue DESC
""")
sql_result.show()

# Verify both plans are identical
print("\nDataFrame API plan:")
(orders.groupBy("category")
    .agg(F.round(F.sum("revenue"),2).alias("total_revenue"),
         F.round(F.avg("unit_price"),2).alias("avg_price"),
         F.countDistinct("product").alias("unique_products"),
         F.count("*").alias("order_count"))
    .orderBy(F.desc("total_revenue"))).explain()

print("\nSpark SQL plan:")
sql_result.explain()

### 2.2 Window Functions

Window functions compute values across a **sliding window of rows** relative to the current row,
without collapsing rows into a single aggregate. They are essential for:
- Ranking (top-N per group)
- Running totals and moving averages
- Lead/lag comparisons (month-over-month change)
- Percentile and quantile analysis

```sql
function() OVER (
    PARTITION BY column     -- defines the group (window frame)
    ORDER BY column         -- defines row ordering within the window
    ROWS/RANGE BETWEEN ...  -- defines the frame boundary
)
```

In [ ]:
from pyspark.sql.window import Window

# ── 1. Ranking: top 3 products by revenue within each category ─────────────
w_cat = Window.partitionBy("category").orderBy(F.desc("revenue"))

top3_per_cat = (
    orders
    .withColumn("rank", F.rank().over(w_cat))
    .filter(F.col("rank") <= 3)
    .select("category","product","revenue","rank")
    .orderBy("category","rank")
)
print("─ Top 3 Orders by Revenue per Category ─")
top3_per_cat.show(15, truncate=False)

In [ ]:
# ── 2. Running total of revenue per city, ordered by date ──────────────────
daily_city = (
    orders
    .groupBy("city", "order_date")
    .agg(F.sum("revenue").alias("daily_revenue"))
)

w_city_date = Window.partitionBy("city").orderBy("order_date")                     .rowsBetween(Window.unboundedPreceding, Window.currentRow)

running = daily_city.withColumn("running_total",
                                F.round(F.sum("daily_revenue").over(w_city_date), 2))

print("─ Running Revenue Total per City (first 12 rows for Amman) ─")
running.filter(F.col("city") == "Amman")        .orderBy("order_date").show(12)

In [ ]:
# ── 3. Customer spending percentile ────────────────────────────────────────
customer_spend = orders.groupBy("customer_id").agg(
    F.round(F.sum("revenue"), 2).alias("total_spend"),
    F.count("*").alias("order_count")
)

w_global = Window.orderBy(F.desc("total_spend"))

customer_ranked = customer_spend     .withColumn("spend_rank",    F.rank().over(w_global))     .withColumn("spend_pctile",  F.round(F.percent_rank().over(w_global) * 100, 1))     .withColumn("spend_tier",
        F.when(F.col("spend_pctile") >= 90, "Top 10%")
         .when(F.col("spend_pctile") >= 75, "Top 25%")
         .when(F.col("spend_pctile") >= 50, "Top 50%")
         .otherwise("Bottom 50%"))

print("─ Top 10 Customers by Total Spend ─")
customer_ranked.orderBy("spend_rank").show(10)

print("─ Customer Tier Distribution ─")
customer_ranked.groupBy("spend_tier").count()     .withColumn("pct", F.round(F.col("count")/customer_ranked.count()*100,1))     .orderBy(F.desc("count")).show()

### Section 2 — Exercises

**Exercise 2.1 — Multi-Dimensional Analysis**

Using the `orders` dataset, write a single DataFrame pipeline (no Spark SQL) that:

1. Computes, for each `city` and `category` combination:
   - Total revenue
   - Average order value
   - Number of unique customers
   - Percentage contribution of that combination to the city's total revenue

2. Filters to retain only combinations where total revenue exceeds 5,000 JOD
3. Ranks combinations within each city by total revenue
4. Shows the top 2 per city

Write the same query using `spark.sql(...)` and verify both produce identical results using `.explain()`.

---

**Exercise 2.2 — Window Function Challenge**

1. Using window functions, compute a **7-day moving average** of daily revenue for each city.
   (`Window.rowsBetween(-6, 0)` gives you the current row plus 6 preceding rows)

2. Identify the **month** (YYYY-MM) with the highest revenue spike per category, defined as:
   the month where actual revenue exceeded the previous month's revenue by the largest percentage.
   Use `F.lag()` to get the previous month's value.

3. For each customer, compute the **time between orders** in days (using `F.lag()` on `order_date`).
   Report customers whose average inter-order gap is less than 7 days (high-frequency buyers).

In [ ]:
# ── Exercise 2.1 & 2.2 workspace ─────────────────────────────────────────────
# Exercise 2.1
# TODO: city+category revenue breakdown with percentage contribution

# Exercise 2.2 — 7-day moving average
daily_revenue = (
    orders.groupBy("city","order_date")
          .agg(F.sum("revenue").alias("daily_revenue"))
)

w_7day = Window.partitionBy("city").orderBy("order_date").rowsBetween(-6, 0)

# TODO: withColumn("moving_avg_7d", ...)
# TODO: show results for one city

# Exercise 2.2 — month-over-month spike
# TODO: groupBy year-month, lag() to get previous month, compute pct change

# Exercise 2.2 — inter-order gap
# TODO: customer-level order history, lag() on order_date, datediff()

---
<h2 style="color:#E25822;font-family:Arial;">Section 3 — Structured Streaming</h2>

### Learning Objectives
- Explain the micro-batch execution model and its trade-offs vs true streaming
- Implement streaming pipelines with file and rate sources
- Design tumbling and sliding window aggregations on event-time
- Apply watermarking to bound state size and handle late-arriving data
- Build a real-world streaming ETL pipeline

### 3.1 Batch vs Stream Processing

| Dimension | Batch | Stream |
|-----------|-------|--------|
| Data model | Bounded dataset | Unbounded, continuously arriving |
| Execution trigger | Scheduled (hourly, nightly) | Continuous, per micro-batch |
| Latency | Minutes to hours | Milliseconds to seconds |
| State management | Stateless (full recompute) | Stateful (running aggregates) |
| Fault model | Rerun failed job | Exactly-once via checkpoints + WAL |
| Complexity | Lower | Higher (ordering, late data, backpressure) |

### 3.2 Structured Streaming Architecture

Spark Structured Streaming models a stream as an **unbounded table** to which new rows are
continuously appended. A streaming query is simply a query over this table, executed incrementally.

```
Unbounded Input Table (stream)          Result Table (sink)
┌────────────────────────────┐          ┌──────────────────────┐
│ row 1  │ ts=10:00:01       │          │ category │ count     │
│ row 2  │ ts=10:00:01       │ ──────▶  │ click    │ 1203      │
│ row 3  │ ts=10:00:02       │  query   │ purchase │  241      │
│  ...   │                   │          │  ...     │           │
└────────────────────────────┘          └──────────────────────┘
         (grows continuously)            (updated each trigger)
```

**Micro-batch execution:** The engine periodically (every N seconds) collects new input rows,
processes them as a mini-batch, and updates the result table. The trigger interval controls
the latency vs throughput trade-off.

**Output modes** determine which rows the engine writes to the sink each trigger:

| Mode | Behavior | Constraint |
|------|----------|-----------|
| `append` | Write only new rows added since last trigger | Cannot be used with aggregations that can update |
| `complete` | Rewrite the entire result table | Only for aggregations |
| `update` | Write only rows that changed since last trigger | Supported for most aggregations |

### 3.3 Event Time vs Processing Time

This distinction is critical for correctness in streaming systems:

- **Processing time** — the wall-clock time when Spark processes the event (system time)
- **Event time** — the timestamp embedded in the event itself (when it actually occurred)

Window operations should almost always use **event time**, not processing time.
Using processing time means events that arrive late (due to network delay, retries) will
fall into the wrong window — a correctness bug, not just a performance issue.

In [ ]:
import time, os, shutil, random, csv
from datetime import datetime, timedelta
from pyspark.sql import functions as F
from pyspark.sql.types import *

# Clean up any leftover checkpoints from previous runs
for d in ["/tmp/chk_s3_rate", "/tmp/chk_s3_file", "/tmp/chk_s3_win", "/tmp/chk_s3_wm", "/tmp/chk_s3_etl"]:
    if os.path.exists(d): shutil.rmtree(d)

print("Streaming section setup complete.")

### 3.4 Building a Streaming Pipeline — Rate Source

The **rate source** generates rows at a configurable rate and is the standard tool for
developing and testing streaming logic without external infrastructure.

Each row contains:
- `timestamp` — processing-time timestamp of when the row was generated
- `value` — monotonically increasing 64-bit integer (0, 1, 2, ...)

The `foreachBatch` sink lets you write arbitrary DataFrame code for each micro-batch,
which is ideal for notebooks and complex multi-sink scenarios.

In [ ]:
# IoT sensor simulation: map the rate source to realistic sensor readings
SENSORS  = [f"S{i:03d}" for i in range(1,11)]    # 10 sensors
PLANTS   = ["PlantA","PlantB","PlantC"]

sensor_array = F.array([F.lit(s) for s in SENSORS])
plant_array  = F.array([F.lit(p) for p in PLANTS])

sensor_stream = (
    spark.readStream.format("rate").option("rowsPerSecond", 6).load()
    .withColumn("sensor_id",   F.element_at(sensor_array,  (F.col("value") % 10 + 1).cast(IntegerType())))
    .withColumn("plant",       F.element_at(plant_array,   (F.col("value") % 3  + 1).cast(IntegerType())))
    .withColumn("temperature", F.round(F.when(F.col("value") % 20 == 0,
                                              F.lit(85.0) + (F.col("value") % 10).cast(DoubleType()))
                                        .otherwise(F.rand() * 30 + 55), 2))
    .withColumn("humidity",    F.round(F.rand() * 40 + 40, 2))
    .withColumnRenamed("timestamp","event_time")
    .drop("value")
)

print("Sensor stream schema:")
sensor_stream.printSchema()

In [ ]:
batches_collected = []

def process_sensor_batch(batch_df, batch_id):
    n = batch_df.count()
    if n == 0: return
    batches_collected.append(batch_id)

    anomalies = batch_df.filter(F.col("temperature") > 80)
    an_count  = anomalies.count()

    print(f"\n{'─'*55}")
    print(f"  Batch {batch_id} | {n} readings | {an_count} anomalies")
    print(f"{'─'*55}")

    # Anomaly alerts
    if an_count > 0:
        print("  ⚠ HIGH TEMPERATURE ALERTS:")
        anomalies.select("sensor_id","plant","temperature","event_time").show(truncate=False)

    # Per-sensor summary
    print("  Per-plant summary:")
    (batch_df
     .groupBy("plant")
     .agg(F.round(F.avg("temperature"),2).alias("avg_temp_C"),
          F.max("temperature").alias("max_temp_C"),
          F.count("*").alias("readings"))
     .orderBy("plant")
     .show(truncate=False))

q = (sensor_stream.writeStream
     .outputMode("append")
     .foreachBatch(process_sensor_batch)
     .trigger(processingTime="4 seconds")
     .option("checkpointLocation","/tmp/chk_s3_rate")
     .start())

print("Sensor stream running for 14 seconds...")
time.sleep(14)
q.stop()
print(f"Done. Processed {len(batches_collected)} batches.")

### 3.5 Window Operations on Event Time

#### Tumbling Windows

Tumbling windows partition time into **non-overlapping, contiguous intervals** of fixed size.
Every event belongs to exactly one window.

```
Window size = 10 seconds

│──── Window 1 ────│──── Window 2 ────│──── Window 3 ────│
│ [00:00 – 00:10)  │ [00:10 – 00:20)  │ [00:20 – 00:30)  │
  e1, e2, e4           e3, e5              e6, e7, e8
```

#### Sliding Windows

Sliding windows have fixed **size** but advance by a smaller **slide** interval,
producing **overlapping** windows. Events may belong to multiple windows.

```
Window size = 20s, slide = 10s

│──────── W1 ────────│
│ [00:00 – 00:20)    │──────── W2 ────────│
                     │ [00:10 – 00:30)    │──────── W3 ────────│
                                          │ [00:20 – 00:40)    │
  e1,e2,e4,e3,e5         e3,e5,e6,e7         e6,e7,e8
```

#### Session Windows (Spark 3.2+)

Session windows group events that occur within a **gap duration** of each other.
Window size is dynamic — determined by activity, not a fixed interval.
Useful for user session analytics, where "sessions" have variable duration.

In [ ]:
# Tumbling window: count anomalies per 10-second window per plant
w_stream = (
    spark.readStream.format("rate").option("rowsPerSecond",8).load()
    .withColumn("plant",       F.element_at(plant_array,  (F.col("value") % 3 + 1).cast(IntegerType())))
    .withColumn("temperature", F.when(F.col("value") % 15 == 0, F.lit(90.0))
                                .otherwise(F.round(F.rand()*30+55, 2)))
    .withColumnRenamed("timestamp","event_time")
)

tumbling_agg = (
    w_stream
    .groupBy(F.window("event_time","10 seconds"), "plant")
    .agg(
        F.count("*").alias("readings"),
        F.sum(F.when(F.col("temperature") > 80, 1).otherwise(0)).alias("anomalies"),
        F.round(F.avg("temperature"),2).alias("avg_temp")
    )
    .select(
        F.col("window.start").alias("win_start"),
        F.col("window.end").alias("win_end"),
        "plant","readings","anomalies","avg_temp"
    )
    .orderBy("win_start","plant")
)

def show_windows(df, bid):
    if df.count() == 0: return
    print(f"\n── Tumbling Window Batch {bid} ──")
    df.show(20, truncate=False)

q_win = (tumbling_agg.writeStream.outputMode("complete")
         .foreachBatch(show_windows)
         .trigger(processingTime="6 seconds")
         .option("checkpointLocation","/tmp/chk_s3_win").start())

print("Tumbling window query — 18 seconds...")
time.sleep(18)
q_win.stop()
print("Done.")

### 3.6 Watermarking and Late Data

#### The Late Data Problem

In a distributed system, events rarely arrive in strict event-time order.
Consider a mobile sensor whose data is buffered when offline and uploaded in bulk:

```
Event time:    10:00:03    10:00:07    10:00:15    10:00:03  ← LATE (was offline)
Arrival time:  10:00:05    10:00:09    10:00:17    10:00:45  ← arrives 42 seconds late
```

Without a strategy to handle late data, Spark must keep all past window state in memory
indefinitely — eventually causing Out-Of-Memory errors.

#### Watermark Definition

A **watermark** is a monotonically increasing lower bound on event time. Formally:

> Given a watermark threshold Δ, at any processing time *t*, Spark guarantees that
> all events with `event_time < max_event_time_seen − Δ` have been received.
> Events arriving with `event_time < watermark` are considered late and **dropped**.

```python
stream.withWatermark("event_time", "Δ")
```

**State management benefit:** Once the watermark advances past a window's end time,
that window's state can be finalized and freed from memory. This gives Spark a **bounded
state guarantee** — critical for long-running streaming jobs.

#### Trade-off: Δ (threshold) selection

| Small Δ | Large Δ |
|---------|---------|
| Less memory (state freed sooner) | More memory (state held longer) |
| More late events dropped | Fewer late events dropped |
| Lower latency results | Higher latency results |

Set Δ based on your observed late-arrival distribution, typically `2 × 99th_percentile_latency`.

In [ ]:
# Watermark demonstration: inject 30-second-late events, set 15s watermark
# Expected: late events are DROPPED (30s > 15s threshold)

late_stream = (
    spark.readStream.format("rate").option("rowsPerSecond",5).load()
    .withColumn("is_late", (F.col("value") % 8 == 0))
    .withColumn("event_time",
        F.when(F.col("is_late"),
               F.col("timestamp") - F.expr("INTERVAL 30 SECONDS"))  # 30s late
         .otherwise(F.col("timestamp")))
    .withColumn("event_type", F.when(F.col("is_late"), F.lit("LATE")).otherwise(F.lit("normal")))
    .drop("timestamp")
)

wm_agg = (
    late_stream
    .withWatermark("event_time","15 seconds")          # accept up to 15s late
    .groupBy(F.window("event_time","10 seconds"), "event_type")
    .agg(F.count("*").alias("count"))
    .select(F.col("window.start").alias("win_start"), "event_type","count")
    .orderBy("win_start","event_type")
)

def show_wm(df, bid):
    if df.count() == 0: return
    print(f"\n── Watermark Batch {bid} ──")
    print("  (LATE rows with event_time 30s ago should not appear — watermark=15s drops them)")
    df.show(20, truncate=False)

q_wm = (wm_agg.writeStream.outputMode("update")
        .foreachBatch(show_wm)
        .trigger(processingTime="5 seconds")
        .option("checkpointLocation","/tmp/chk_s3_wm").start())

print("Watermark demo — 15 seconds...")
time.sleep(15)
q_wm.stop()
print("Done. Only 'normal' rows should appear in the output above.")

### 3.7 Real-World Pattern — Streaming ETL with Enrichment

Production streaming pipelines typically combine:
1. **Ingestion** — read from Kafka or file drop zone
2. **Validation / filtering** — reject malformed or out-of-range events
3. **Enrichment** — join stream with a static lookup (broadcast)
4. **Aggregation** — windowed or running totals
5. **Multi-sink output** — write alerts to one sink, summaries to another

The `foreachBatch` pattern makes this possible in a single query.

In [ ]:
# ── Static enrichment table: sensor → plant metadata ────────────────────────
meta_data = [(f"S{i:03d}", f"Plant{'ABC'[i%3]}", ["Assembly","Cooling","Packaging"][(i*3)%3], i*2.5)
             for i in range(1,11)]
sensor_meta = spark.createDataFrame(meta_data,
    ["sensor_id","plant","process_line","threshold_C"])
sensor_meta.cache()
print("Sensor metadata:")
sensor_meta.show()

In [ ]:
# ── Streaming ETL pipeline ────────────────────────────────────────────────────
etl_stream = (
    spark.readStream.format("rate").option("rowsPerSecond",8).load()
    .withColumn("sensor_id",   F.element_at(sensor_array, (F.col("value") % 10 + 1).cast(IntegerType())))
    .withColumn("temperature", F.round(F.when(F.col("value") % 12 == 0, F.lit(88.0) + F.rand()*10)
                                        .otherwise(F.rand()*25 + 58), 2))
    .withColumnRenamed("timestamp","event_time")
)

fraud_log, summary_log = [], []

def etl_batch(batch_df, batch_id):
    n = batch_df.count()
    if n == 0: return

    # Step 1: Enrich with sensor metadata (broadcast the small lookup)
    enriched = batch_df.join(F.broadcast(sensor_meta), "sensor_id", "left")

    # Step 2: Validate — flag readings that exceed per-sensor threshold
    enriched = enriched.withColumn("alert",
        F.col("temperature") > F.col("threshold_C"))

    alerts  = enriched.filter(F.col("alert") == True)
    normal  = enriched.filter(F.col("alert") == False)

    print(f"\n{'━'*60}")
    print(f"  ETL Batch {batch_id} | {n} readings | {alerts.count()} alerts")
    print(f"{'━'*60}")

    if alerts.count() > 0:
        print("  ▶ ALERTS (temperature exceeds sensor threshold):")
        alerts.select("sensor_id","plant","process_line",
                      "temperature","threshold_C","event_time").show(5,truncate=False)
        fraud_log.append(alerts.count())

    print("  ▶ Avg temperature by process line (normal readings only):")
    normal.groupBy("process_line")         .agg(F.round(F.avg("temperature"),2).alias("avg_temp"),
             F.count("*").alias("count"))         .orderBy("process_line").show(truncate=False)
    summary_log.append(batch_id)

q_etl = (etl_stream.writeStream.outputMode("append")
         .foreachBatch(etl_batch)
         .trigger(processingTime="5 seconds")
         .option("checkpointLocation","/tmp/chk_s3_etl").start())

print("ETL pipeline running for 20 seconds...")
time.sleep(20)
q_etl.stop()
print(f"\nTotal alert events: {sum(fraud_log)} across {len(summary_log)} batches")

### Section 3 — Exercises

**Exercise 3.1 — Window Design**

You are building a streaming dashboard for a ride-sharing platform.
Events arrive with `driver_id`, `ride_fare` (JOD), `pickup_city`, and `event_time`.

1. Design a streaming query that computes the **total fare collected per city per 5-minute tumbling window**.
   Show the `window_start`, `window_end`, `city`, `total_fare`, and `ride_count`.

2. Modify the query to use a **sliding window** of 10 minutes that slides every 2 minutes.
   How many windows will a single event at time T belong to? Prove this with a formula.

3. Add watermarking with a **3-minute threshold**. Explain in writing:
   - At what point does Spark finalize (close) a window?
   - What happens to a fare event that arrives 4 minutes after its event_time?
   - What happens to an event that arrives 2 minutes after its event_time?

4. Simulate the scenario by generating a rate-based stream that includes late events.
   Verify your watermark behavior by observing which windows appear in the output.

---

**Exercise 3.2 — Streaming ETL Design**

A bank needs a real-time transaction monitoring system.
Transactions have: `txn_id`, `account_id`, `amount`, `merchant_category`, `city`, `event_time`.

1. Implement a streaming pipeline that:
   - Flags any transaction over **5,000 JOD** as a high-value alert
   - Computes total spending per `merchant_category` per **1-minute tumbling window**
   - Joins with a static lookup table mapping `merchant_category` to `risk_level` (Low/Medium/High)
   - Applies a 30-second watermark

2. Use `foreachBatch` to route alerts and summaries to separate print outputs (simulating different sinks).

3. Analyze: if this system processes 10,000 transactions/second and each transaction is 500 bytes,
   what is the data volume per day? How does watermarking help keep memory usage bounded?

In [ ]:
# ── Exercise 3.1 workspace ───────────────────────────────────────────────────
# Ride-sharing fare aggregation
CITIES_RIDE = ["Amman","Irbid","Zarqa","Aqaba","Madaba"]
city_array  = F.array([F.lit(c) for c in CITIES_RIDE])

# TODO: build ride_stream from rate source
# ride_stream = (spark.readStream.format("rate")...

# TODO: 5-minute tumbling window per city

# TODO: 10-minute sliding window, 2-minute slide

# TODO: add watermark, explain finalization logic in a markdown cell

In [ ]:
# ── Exercise 3.2 workspace ───────────────────────────────────────────────────
MERCHANTS = ["Grocery","Restaurant","Gas Station","Electronics","Travel","Healthcare"]
RISK_LEVEL = {"Grocery":"Low","Restaurant":"Low","Gas Station":"Medium",
              "Electronics":"High","Travel":"High","Healthcare":"Low"}

risk_lookup = spark.createDataFrame(
    [(k, v) for k, v in RISK_LEVEL.items()], ["merchant_category","risk_level"])

# TODO: build txn_stream from rate source
# TODO: foreachBatch ETL: flag high-value, join risk_lookup, 1-min window
# TODO: separate alert and summary outputs

---
<h2 style="color:#E25822;font-family:Arial;">Section 4 — Machine Learning with MLlib</h2>

### Learning Objectives
- Explain the pipeline abstraction and why it prevents data leakage
- Engineer features from categorical and numerical inputs
- Train, evaluate, and compare classification and regression models
- Apply K-Means clustering with the Elbow Method and Silhouette Score
- Tune hyperparameters using k-fold cross-validation

### 4.1 The Pipeline Abstraction

MLlib's `Pipeline` is a directed acyclic graph of **Estimators** and **Transformers**:

- **Transformer** — takes a DataFrame and returns a new DataFrame (`transform()`)
  Examples: `VectorAssembler`, `OneHotEncoder`, `StandardScaler` (after fitting)

- **Estimator** — learns parameters from data, produces a Transformer (`fit()`)
  Examples: `StringIndexer`, `StandardScaler`, `RandomForestClassifier`

```
Pipeline.fit(train_df):
  Stage 1: StringIndexer.fit(train_df)  → StringIndexerModel
                StringIndexerModel.transform(train_df) →  df_1
  Stage 2: OneHotEncoder.fit(df_1)      → OneHotEncoderModel
                OneHotEncoderModel.transform(df_1)     →  df_2
  Stage 3: VectorAssembler.transform(df_2)             →  df_3
  Stage 4: RandomForest.fit(df_3)       → RandomForestModel
  Returns: PipelineModel (all fitted stages)

PipelineModel.transform(test_df):
  Applies all fitted stages sequentially to test_df
```

**Why Pipelines prevent data leakage:**
If you fit the scaler on the full dataset before splitting into train/test,
the scaler has "seen" the test data — its mean and std are influenced by test points.
When wrapped in a Pipeline, `fit()` is called only on `train_df`, ensuring the scaler
learns statistics purely from training data.

#### Feature Engineering Reference

| Raw feature type | Encoding strategy | MLlib class |
|-----------------|-------------------|-------------|
| Categorical string | Index → One-hot | `StringIndexer` → `OneHotEncoder` |
| Categorical ordinal | Index only | `StringIndexer` |
| Numerical, different scales | Standardize | `StandardScaler` (zero mean, unit σ) |
| Numerical, bounded | Normalize | `MinMaxScaler` (to [0,1]) |
| Text | TF-IDF | `Tokenizer` → `HashingTF` → `IDF` |
| Multiple columns | Merge | `VectorAssembler` |

In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import (VectorAssembler, StandardScaler, MinMaxScaler,
                                 StringIndexer, OneHotEncoder)
from pyspark.ml.classification import (RandomForestClassifier, LogisticRegression,
                                        GBTClassifier)
from pyspark.ml.regression import (RandomForestRegressor, LinearRegression,
                                    DecisionTreeRegressor)
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import (BinaryClassificationEvaluator,
                                    MulticlassClassificationEvaluator,
                                    RegressionEvaluator, ClusteringEvaluator)
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
print("MLlib imports ready.")

### 4.2 Classification — Telecom Customer Churn

**Business context:** A Jordanian telecom operator wants to predict which customers will
cancel their subscription in the next month. The model will be used to prioritize retention offers.

**Features:**
- `age` — customer age
- `tenure_months` — months as a subscriber
- `monthly_charge` — monthly bill (JOD)
- `num_services` — number of subscribed services (voice, data, TV, etc.)
- `contract_type` — Monthly / One-Year / Two-Year
- `payment_method` — Electronic / Mailed Check / Bank Transfer / Credit Card
- `support_calls` — number of customer support calls in the last 3 months

**Label:** `churned` (1 = cancelled, 0 = stayed)

In [ ]:
random.seed(2024)

CONTRACTS = ["Monthly","One-Year","Two-Year"]
PAYMENTS  = ["Electronic","Mailed Check","Bank Transfer","Credit Card"]

def churn_prob(age, tenure, charge, services, contract, support_calls):
    logit = (- 0.04 * tenure
             + 0.018 * charge
             - 0.25 * services
             + (1.2 if contract=="Monthly" else 0.3 if contract=="One-Year" else -0.5)
             + 0.15 * support_calls
             - 0.01 * age + 0.3)
    return 1 / (1 + math.exp(-logit))

rows = []
for _ in range(2000):
    age       = random.randint(18, 75)
    tenure    = random.randint(1, 84)
    charge    = round(random.uniform(15, 200), 2)
    services  = random.randint(1, 6)
    contract  = random.choice(CONTRACTS)
    payment   = random.choice(PAYMENTS)
    s_calls   = random.randint(0, 10)
    p = churn_prob(age, tenure, charge, services, contract, s_calls)
    churned   = 1 if random.random() < p else 0
    rows.append((age, tenure, charge, services, contract, payment, s_calls, churned))

churn_schema = StructType([
    StructField("age",            IntegerType(), False),
    StructField("tenure_months",  IntegerType(), False),
    StructField("monthly_charge", DoubleType(),  False),
    StructField("num_services",   IntegerType(), False),
    StructField("contract_type",  StringType(),  False),
    StructField("payment_method", StringType(),  False),
    StructField("support_calls",  IntegerType(), False),
    StructField("churned",        IntegerType(), False),
])

churn_df = spark.createDataFrame(rows, schema=churn_schema)
churn_df = churn_df.withColumn("label", F.col("churned").cast(DoubleType()))

print(f"Dataset: {churn_df.count():,} customers")
print("\nClass distribution:")
churn_df.groupBy("churned").count()     .withColumn("pct", F.round(F.col("count")/churn_df.count()*100,1)).show()
print("\nNumerical summary:")
churn_df.select("age","tenure_months","monthly_charge","num_services","support_calls").describe().show()

In [ ]:
# ── Feature engineering + Pipeline ──────────────────────────────────────────
train_df, test_df = churn_df.randomSplit([0.8, 0.2], seed=42)
print(f"Train: {train_df.count():,}  |  Test: {test_df.count():,}")

# Categorical encoding
contract_idx = StringIndexer(inputCol="contract_type",  outputCol="contract_idx")
payment_idx  = StringIndexer(inputCol="payment_method", outputCol="payment_idx")
contract_ohe = OneHotEncoder(inputCol="contract_idx",   outputCol="contract_vec")
payment_ohe  = OneHotEncoder(inputCol="payment_idx",    outputCol="payment_vec")

# Assemble all features
assembler = VectorAssembler(
    inputCols=["age","tenure_months","monthly_charge","num_services",
               "contract_vec","payment_vec","support_calls"],
    outputCol="raw_features")

scaler = StandardScaler(inputCol="raw_features", outputCol="features",
                        withMean=True, withStd=True)

# Classifier
rf = RandomForestClassifier(featuresCol="features", labelCol="label",
                             numTrees=100, maxDepth=6, seed=42)

pipeline = Pipeline(stages=[contract_idx, payment_idx, contract_ohe, payment_ohe,
                              assembler, scaler, rf])

print("\nTraining Random Forest pipeline...")
model = pipeline.fit(train_df)
preds = model.transform(test_df)
print("Done.")

In [ ]:
# ── Comprehensive evaluation ─────────────────────────────────────────────────
def evaluate_classifier(preds, label="label"):
    acc  = MulticlassClassificationEvaluator(labelCol=label, predictionCol="prediction",
                                              metricName="accuracy").evaluate(preds)
    prec = MulticlassClassificationEvaluator(labelCol=label, predictionCol="prediction",
                                              metricName="weightedPrecision").evaluate(preds)
    rec  = MulticlassClassificationEvaluator(labelCol=label, predictionCol="prediction",
                                              metricName="weightedRecall").evaluate(preds)
    f1   = MulticlassClassificationEvaluator(labelCol=label, predictionCol="prediction",
                                              metricName="f1").evaluate(preds)
    auc  = BinaryClassificationEvaluator(labelCol=label,
                                          metricName="areaUnderROC").evaluate(preds)
    return {"Accuracy":acc,"Precision":prec,"Recall":rec,"F1":f1,"ROC-AUC":auc}

metrics = evaluate_classifier(preds)
print("─ Random Forest Evaluation ─")
for k,v in metrics.items():
    bar = "█" * int(v*30)
    print(f"  {k:<12} {v:.4f}  {bar}")

# Confusion matrix
print("\n─ Confusion Matrix ─")
print(f"  {'':15} {'Pred=0':>10} {'Pred=1':>10}")
cm = preds.groupBy("label","prediction").count().collect()
cm_d = {(int(r["label"]),int(r["prediction"])):r["count"] for r in cm}
for actual in [0,1]:
    row = f"  {'Actual='+str(actual):<15}"
    for pred in [0,1]:
        row += f"{cm_d.get((actual,pred),0):>10}"
    print(row)

In [ ]:
# ── Feature importances ─────────────────────────────────────────────────────
rf_model = model.stages[-1]

# Feature name mapping (accounts for OHE expansion)
feat_names = (["age","tenure_months","monthly_charge","num_services"]
              + [f"contract_{c}" for c in ["Monthly","One-Year"]]       # OHE drops last
              + [f"payment_{p}" for p in ["Electronic","Mailed","Bank"]] # OHE drops last
              + ["support_calls"])

importances = rf_model.featureImportances.toArray()
# Pad if feature count doesn't match exactly
n = min(len(feat_names), len(importances))

print("─ Feature Importances (Random Forest) ─")
for name, imp in sorted(zip(feat_names[:n], importances[:n]), key=lambda x:-x[1]):
    bar = "█" * int(imp*50)
    print(f"  {name:<22} {imp:.4f}  {bar}")

In [ ]:
# ── Compare Random Forest vs Logistic Regression vs GBT ────────────────────
results = {}

# Logistic Regression
lr_pipeline = Pipeline(stages=[contract_idx, payment_idx, contract_ohe, payment_ohe,
                                 assembler, scaler,
                                 LogisticRegression(featuresCol="features",
                                                    labelCol="label", maxIter=100)])
lr_model  = lr_pipeline.fit(train_df)
lr_preds  = lr_model.transform(test_df)
results["Logistic Regression"] = evaluate_classifier(lr_preds)

# Gradient Boosted Trees
gbt_pipeline = Pipeline(stages=[contract_idx, payment_idx, contract_ohe, payment_ohe,
                                  assembler, scaler,
                                  GBTClassifier(featuresCol="features",
                                                labelCol="label", maxIter=50)])
gbt_model = gbt_pipeline.fit(train_df)
gbt_preds = gbt_model.transform(test_df)
results["GBT Classifier"] = evaluate_classifier(gbt_preds)
results["Random Forest"]  = metrics

print(f"\n{'Model':<25} {'Accuracy':>9} {'F1':>9} {'ROC-AUC':>9}")
print("─" * 56)
for model_name, m in results.items():
    print(f"{model_name:<25} {m['Accuracy']:>9.4f} {m['F1']:>9.4f} {m['ROC-AUC']:>9.4f}")

In [ ]:
# ── Hyperparameter tuning with k-fold cross-validation ──────────────────────
# Tune Random Forest: numTrees and maxDepth
# 3 × 2 = 6 parameter combinations × 3 folds = 18 model fits

cv_rf = RandomForestClassifier(featuresCol="features", labelCol="label", seed=42)
cv_pipeline = Pipeline(stages=[contract_idx, payment_idx, contract_ohe, payment_ohe,
                                 assembler, scaler, cv_rf])

param_grid = (ParamGridBuilder()
    .addGrid(cv_rf.numTrees,  [50, 100, 150])
    .addGrid(cv_rf.maxDepth,  [4, 8])
    .build())

cross_val = CrossValidator(
    estimator=cv_pipeline,
    estimatorParamMaps=param_grid,
    evaluator=BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderROC"),
    numFolds=3, seed=42)

print("Running 3-fold cross-validation (6 parameter combinations × 3 folds = 18 fits)...")
cv_model = cross_val.fit(train_df)

best_rf   = cv_model.bestModel.stages[-1]
cv_preds  = cv_model.transform(test_df)
cv_metrics = evaluate_classifier(cv_preds)

print(f"\nBest hyperparameters:")
print(f"  numTrees : {best_rf.getNumTrees}")
print(f"  maxDepth : {best_rf.getMaxDepth()}")
print(f"\nCV best model — Test set:")
for k,v in cv_metrics.items():
    print(f"  {k:<12}: {v:.4f}")

print("\nCV average AUC per parameter combination:")
for i,(params,avg) in enumerate(zip(param_grid, cv_model.avgMetrics)):
    n=params[cv_rf.numTrees]; d=params[cv_rf.maxDepth]
    print(f"  [{i+1}] numTrees={n:>3}, maxDepth={d}  →  AUC={avg:.4f}")

### 4.3 Regression — Apartment Price Prediction

**Business context:** Predict residential apartment prices in Jordan for a real estate portal.

**Features:** `city`, `bedrooms`, `bathrooms`, `area_m2`, `floor`, `age_years`, `has_parking`

In [ ]:
random.seed(99)
CITIES_APT = ["Amman","Irbid","Zarqa","Aqaba","Madaba"]
BASE_PRICE  = {"Amman":900,"Irbid":550,"Zarqa":420,"Aqaba":650,"Madaba":480}

apt_rows = []
for _ in range(1500):
    city     = random.choice(CITIES_APT)
    beds     = random.randint(1,6)
    baths    = random.randint(1,4)
    area     = random.randint(60,300)
    floor    = random.randint(0,15)
    age_yrs  = random.randint(0,40)
    parking  = random.randint(0,1)
    price    = (BASE_PRICE[city]*area + beds*4000 + baths*2500
                - age_yrs*300 + floor*200 + parking*8000
                + random.gauss(0,12000))
    price    = max(25000.0, round(price,-2))
    apt_rows.append((city,beds,baths,area,floor,age_yrs,parking,price))

apt_df = spark.createDataFrame(apt_rows,
    ["city","bedrooms","bathrooms","area_m2","floor","age_years","has_parking","price"])

city_idx  = StringIndexer(inputCol="city", outputCol="city_idx")
city_ohe  = OneHotEncoder(inputCol="city_idx", outputCol="city_vec")
apt_asm   = VectorAssembler(
    inputCols=["city_vec","bedrooms","bathrooms","area_m2","floor","age_years","has_parking"],
    outputCol="features")

train_apt, test_apt = apt_df.randomSplit([0.8,0.2], seed=42)

regressors = {
    "Linear Regression":    LinearRegression(featuresCol="features",labelCol="price"),
    "Decision Tree":        DecisionTreeRegressor(featuresCol="features",labelCol="price",maxDepth=6),
    "Random Forest":        RandomForestRegressor(featuresCol="features",labelCol="price",numTrees=50,seed=42),
}

rmse_eval = RegressionEvaluator(labelCol="price",metricName="rmse")
r2_eval   = RegressionEvaluator(labelCol="price",metricName="r2")
mae_eval  = RegressionEvaluator(labelCol="price",metricName="mae")

print(f"{'Model':<25} {'RMSE (JOD)':>12} {'MAE (JOD)':>12} {'R²':>8}")
print("─"*62)

for name, reg in regressors.items():
    pipe = Pipeline(stages=[city_idx, city_ohe, apt_asm, reg])
    m    = pipe.fit(train_apt)
    p    = m.transform(test_apt)
    print(f"{name:<25} {rmse_eval.evaluate(p):>12,.0f} {mae_eval.evaluate(p):>12,.0f} {r2_eval.evaluate(p):>8.4f}")

### 4.4 Clustering — Customer Segmentation

**Business context:** Segment customers of the e-commerce orders dataset into behavioral groups
for targeted marketing campaigns.

In [ ]:
# Compute customer-level features from the orders dataset
customer_features = (
    orders
    .groupBy("customer_id")
    .agg(
        F.count("*")                       .alias("order_count"),
        F.round(F.sum("revenue"),2)        .alias("total_spend"),
        F.round(F.avg("revenue"),2)        .alias("avg_order_value"),
        F.countDistinct("category")        .alias("categories_bought"),
        F.round(F.stddev("revenue"),2)     .alias("spend_stddev"),
    )
    .fillna({"spend_stddev":0.0})
)

print(f"Customer features: {customer_features.count():,} customers")
customer_features.describe().show()

# Assemble and normalize features for KMeans
km_asm = VectorAssembler(
    inputCols=["order_count","total_spend","avg_order_value","categories_bought","spend_stddev"],
    outputCol="raw_feat")
km_scaler = MinMaxScaler(inputCol="raw_feat", outputCol="features")

prep_pipeline = Pipeline(stages=[km_asm, km_scaler])
prep_model    = prep_pipeline.fit(customer_features)
feat_df       = prep_model.transform(customer_features)

sil_eval = ClusteringEvaluator(featuresCol="features", predictionCol="cluster",
                                metricName="silhouette", distanceMeasure="squaredEuclidean")

print("\nElbow Method & Silhouette Score:")
print(f"  {'k':>3}  {'Silhouette':>12}  {'WCSSE':>14}")
print("  " + "─"*35)
for k in range(2,9):
    km = KMeans(featuresCol="features",predictionCol="cluster",k=k,seed=42,maxIter=30)
    km_m = km.fit(feat_df)
    km_p = km_m.transform(feat_df)
    wcsse = km_m.summary.trainingCost
    sil   = sil_eval.evaluate(km_p)
    print(f"  {k:>3}  {sil:>12.4f}  {wcsse:>14,.2f}")

In [ ]:
# Best k analysis and cluster profiling
best_k  = 4
km_best = KMeans(featuresCol="features",predictionCol="cluster",k=best_k,seed=42,maxIter=30)
km_model = km_best.fit(feat_df)
clustered = km_model.transform(feat_df)

print(f"\nCluster Profiles (k={best_k}):")
(clustered
 .groupBy("cluster")
 .agg(
     F.count("*")                        .alias("customer_count"),
     F.round(F.avg("order_count"),1)     .alias("avg_orders"),
     F.round(F.avg("total_spend"),0)     .alias("avg_total_spend_JOD"),
     F.round(F.avg("avg_order_value"),0) .alias("avg_order_value_JOD"),
     F.round(F.avg("categories_bought"),1).alias("avg_categories"),
 )
 .orderBy(F.desc("avg_total_spend_JOD"))
 .show())

print("▶ Interpret each cluster: What marketing strategy suits each group?")

### Section 4 — Exercises

**Exercise 4.1 — Model Selection and Evaluation**

1. Train a **Logistic Regression** and a **GBT Classifier** on the churn dataset.
   For each model, report: Accuracy, Precision, Recall, F1, ROC-AUC, and the Confusion Matrix.

2. In a churn prediction context, a **False Negative** (predicting "stays" when the customer actually churns)
   and a **False Positive** (predicting "churns" when the customer stays) have different business costs.
   Assume:
   - Cost of a missed churner (False Negative): 200 JOD (lost subscription revenue)
   - Cost of an unnecessary retention offer (False Positive): 15 JOD
   Using your confusion matrix, compute the **total business cost** for each model.
   Which model would you deploy? Justify quantitatively.

3. The dataset has a class imbalance (more non-churners than churners).
   Explain how this affects the Accuracy metric specifically. Why is F1 or ROC-AUC more reliable here?

---

**Exercise 4.2 — Feature Engineering Impact**

1. Train a Random Forest classifier on the churn dataset using **only the numerical features**
   (age, tenure_months, monthly_charge, num_services, support_calls). Record the ROC-AUC.

2. Train the same model **including the categorical features** (contract_type, payment_method).
   Record the ROC-AUC.

3. Compute the ROC-AUC improvement and explain which categorical feature contributes more
   (use feature importances to support your argument).

---

**Exercise 4.3 — Cluster Interpretation**

After running K-Means with k=4 on the customer segments:

1. Profile each cluster using at least 5 behavioral metrics.
2. Give each cluster a descriptive business name (e.g., "High-Value Frequent Buyers").
3. For each cluster, propose one specific marketing action and justify it using the cluster profile.
4. Run the silhouette analysis for k=3 and k=5. Explain why k=4 was or was not the best choice.

In [ ]:
# ── Exercise 4.1 workspace ───────────────────────────────────────────────────
# TODO: train LR and GBT, compute confusion matrix for each
# TODO: compute business cost = FN*200 + FP*15 for each model
# TODO: write recommendation in markdown

# ── Exercise 4.2 workspace ───────────────────────────────────────────────────
# Numerical-only model
num_only_asm = VectorAssembler(
    inputCols=["age","tenure_months","monthly_charge","num_services","support_calls"],
    outputCol="features_num")
# TODO: build pipeline, train, evaluate, compare

# ── Exercise 4.3 workspace ───────────────────────────────────────────────────
# TODO: profile clusters with more metrics
# TODO: name clusters and propose actions

---
<h2 style="color:#E25822;font-family:Arial;">Section 5 — Performance Optimization</h2>

### Learning Objectives
- Quantify the cost of shuffles and explain why they dominate job runtime
- Apply caching and persistence strategies correctly, including storage level selection
- Tune partition count for different cluster configurations
- Use broadcast joins to eliminate shuffle in asymmetric joins
- Read and interpret the Spark UI and query plans to diagnose bottlenecks
- Explain Adaptive Query Execution (AQE) and its runtime optimizations

### 5.1 Shuffle Cost Analysis

A **shuffle** is the redistribution of data across partitions — required by all wide transformations.
It is the dominant performance cost in most Spark jobs. Understanding its mechanics is essential
for writing efficient code.

**Shuffle write phase (map side):**
1. Each task writes its output records to local disk, partitioned and sorted by destination partition key
2. The number of output files = `num_map_tasks × spark.sql.shuffle.partitions`

**Shuffle read phase (reduce side):**
1. Each reduce task fetches its partition from all map tasks over the network
2. Results are merged and sorted (sort-merge join) or aggregated (hash aggregation)

**Total I/O cost of a shuffle:**
- Disk write: map output written to executor local disk
- Network transfer: all data crosses the network
- Disk read: reduce tasks read from remote executors
- CPU: sort + merge

```
For a 100 GB dataset with 200 shuffle partitions:
  - Each of 200 reduce tasks fetches ~500 MB from across the cluster
  - Total network bytes transferred ≈ 100 GB
  - Plus disk I/O on both sides
```

#### Minimizing Shuffles — Design Patterns

| Pattern | Benefit |
|---------|---------|
| Filter before groupBy/join | Reduce data volume before expensive shuffle |
| Broadcast small tables | Eliminate join shuffle entirely |
| Repartition once, reuse | Pay shuffle cost once for multiple downstream operations |
| Use `reduceByKey` not `groupByKey` (RDD) | Partial aggregation before shuffle |
| Partition by join key | Co-locate data for repeated joins on same key |

In [ ]:
import time
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.functions import broadcast
from pyspark.storagelevel import StorageLevel
import os, glob

TMP = "/tmp/spark_module5"
os.makedirs(TMP, exist_ok=True)

def make_orders(n=2_000_000):
    """Synthetic orders DataFrame — mimics a realistic fact table."""
    return (
        spark.range(n)
        .withColumn("customer_id",  (F.col("id") % 10000).cast(IntegerType()))
        .withColumn("product_id",   (F.col("id") % 200).cast(IntegerType()))
        .withColumn("category",     F.element_at(
            F.array(*[F.lit(c) for c in ["Electronics","Clothing","Food","Sports","Books"]]),
            (F.col("id") % 5 + 1).cast(IntegerType())))
        .withColumn("revenue",      F.round(F.rand(seed=7)*500+5, 2))
        .withColumn("quantity",     (F.rand(seed=3)*10+1).cast(IntegerType()))
    )

df_orders = make_orders()
print(f"Orders DataFrame: {df_orders.rdd.getNumPartitions()} partitions")

In [ ]:
# ── Demonstrating shuffle cost ───────────────────────────────────────────────
# Count partitions before and after a wide transformation

print(f"Before groupBy: {df_orders.rdd.getNumPartitions()} partitions")
grouped = df_orders.groupBy("category").agg(F.sum("revenue"))
print(f"After groupBy:  {grouped.rdd.getNumPartitions()} partitions")
print(f"  (controlled by spark.sql.shuffle.partitions = {spark.conf.get('spark.sql.shuffle.partitions')})")

# Measure filter early vs filter late
df_large = make_orders(3_000_000)

# BAD: join/group on full 3M rows, then filter
t0 = time.time()
(df_large.groupBy("category","customer_id")
 .agg(F.sum("revenue").alias("total"))
 .filter(F.col("category")=="Electronics")
 .count())
t_late = time.time() - t0

# GOOD: filter first, then group on the reduced set (~600K rows)
t0 = time.time()
(df_large.filter(F.col("category")=="Electronics")
 .groupBy("category","customer_id")
 .agg(F.sum("revenue").alias("total"))
 .count())
t_early = time.time() - t0

print(f"\nFilter AFTER groupBy  : {t_late:.2f}s")
print(f"Filter BEFORE groupBy : {t_early:.2f}s")
print(f"Speedup               : {t_late/t_early:.1f}×")
print("(Catalyst often pushes filters automatically, but always write them first in practice)")

### 5.2 Caching and Persistence

#### When to Cache

Cache a DataFrame when **all three** conditions hold:
1. The DataFrame is **expensive to compute** (multiple stages, large source)
2. The DataFrame will be **accessed more than once** (multiple downstream actions)
3. It **fits in available memory** (or the spill-to-disk cost is acceptable)

Do **not** cache DataFrames that are accessed only once — this wastes memory and
adds cache serialization overhead.

#### Storage Levels

| Level | RAM | Disk | Serialized | Replicated | Best use |
|-------|:---:|:----:|:----------:|:----------:|---------|
| `MEMORY_ONLY` | ✓ | ✗ | ✗ | ✗ | Small DFs; fastest access |
| `MEMORY_AND_DISK` | ✓ | ✓ | ✗ | ✗ | **Default** (`cache()`); safe general choice |
| `MEMORY_ONLY_SER` | ✓ | ✗ | ✓ | ✗ | Reduce GC pressure with Kryo serializer |
| `DISK_ONLY` | ✗ | ✓ | ✓ | ✗ | Very large DFs; RAM scarce |
| `MEMORY_AND_DISK_2` | ✓ | ✓ | ✗ | ✓ | Fault-tolerant clusters; 2× storage cost |

`df.cache()` is equivalent to `df.persist(StorageLevel.MEMORY_AND_DISK)`.

#### Cache Lifecycle

```
df.cache()        → marks for caching (lazy)
df.count()        → triggers first action; populates cache
df.groupBy(...)   → cache hit; no recomputation
df.unpersist()    → explicitly frees memory (do this when done!)
```

Without `unpersist()`, cached DataFrames remain in memory until the SparkSession ends
or the executor is evicted under memory pressure — potentially causing OOM errors.

In [ ]:
# ── Cache benchmark ─────────────────────────────────────────────────────────
N = 5_000_000

def make_df(n=N):
    return (spark.range(n)
            .withColumn("val",  F.rand(seed=7)*1000)
            .withColumn("cat",  (F.col("id") % 20).cast(StringType()))
            .withColumn("score",F.randn(seed=13)*50+100))

# WITHOUT cache: 3 independent actions each recompute from scratch
df_raw = make_df()
t0 = time.time()
_ = df_raw.groupBy("cat").agg(F.avg("val")).count()
_ = df_raw.groupBy("cat").agg(F.sum("score")).count()
_ = df_raw.filter(F.col("val")>800).count()
t_no_cache = time.time() - t0

# WITH cache: populate once, reuse 3 times
df_c = make_df()
df_c.persist(StorageLevel.MEMORY_AND_DISK)
t_populate = time.time()
_ = df_c.count()   # populate cache
t_populate = time.time() - t_populate

t0 = time.time()
_ = df_c.groupBy("cat").agg(F.avg("val")).count()
_ = df_c.groupBy("cat").agg(F.sum("score")).count()
_ = df_c.filter(F.col("val")>800).count()
t_cached = time.time() - t0

df_c.unpersist()

print(f"{'Metric':<35} {'Value':>10}")
print("─"*47)
print(f"{'Cache population (first action)':<35} {t_populate:>9.2f}s")
print(f"{'3 actions WITHOUT cache':<35} {t_no_cache:>9.2f}s")
print(f"{'3 actions WITH cache':<35} {t_cached:>9.2f}s")
print(f"{'Net speedup (3 actions only)':<35} {t_no_cache/t_cached:>9.1f}×")
print(f"{'Break-even (cache amortized)':<35} {'Yes' if (t_populate+t_cached)<t_no_cache else 'No':>10}")

### 5.3 Partitioning

#### Partition Count Guidelines

The number of partitions directly controls the degree of parallelism — one task per partition,
one task per CPU core at a time.

```
Too few partitions:   Cores sit idle
                      ┌─────────────────────────────────┐
                      │ Core 1: ████████████████████████ │ (busy)
                      │ Core 2: ░░░░░░░░░░░░░░░░░░░░░░░░ │ (idle)
                      │ Core 3: ░░░░░░░░░░░░░░░░░░░░░░░░ │ (idle)
                      └─────────────────────────────────┘

Optimal partitions:   All cores busy
                      ┌─────────────────────────────────┐
                      │ Core 1: ████████ ████████        │
                      │ Core 2: ████████ ████████        │
                      │ Core 3: ████████ ████████        │
                      └─────────────────────────────────┘

Too many partitions:  Task scheduling overhead dominates
                      ┌─────────────────────────────────┐
                      │ Core 1: ██░██░██░██░██░██░██░██░ │ (many tiny tasks)
                      └─────────────────────────────────┘
```

**Rule of thumb:** `2–4 × num_cores` partitions, with each partition **128 MB – 512 MB**.

#### repartition() vs coalesce()

| | `repartition(n)` | `coalesce(n)` |
|--|-----------------|---------------|
| Allows increase | ✓ | ✗ |
| Allows decrease | ✓ | ✓ |
| Triggers shuffle | Always | Never (narrow) |
| Data balance | Perfectly balanced | May be uneven |
| Use case | Increase partitions OR need even balance | Reduce partitions cheaply before write |

#### spark.sql.shuffle.partitions

This setting controls the number of partitions **after every shuffle** (groupBy, join, etc.).
The default of 200 was designed for large clusters. On a small cluster or local machine,
200 produces tiny partitions with excessive scheduling overhead.

**Set this to 2–4× your total core count** at session creation.

In [ ]:
CORES = spark.sparkContext.defaultParallelism
print(f"Available CPU cores: {CORES}")

df_bench = make_df(1_000_000)

# Test different partition counts
print("\nEffect of partition count on groupBy performance:")
print(f"  {'Partitions':<14} {'Time':>8} {'Notes'}")
print("  " + "─"*45)

for n_parts, notes in [
    (1,        "too few — all work on 1 core"),
    (CORES,    "1 per core"),
    (CORES*2,  "2 per core (good)"),
    (CORES*4,  "4 per core (good)"),
    (50,       "too many — task overhead"),
]:
    df_t = df_bench.repartition(n_parts)
    t0 = time.time()
    df_t.groupBy("cat").agg(F.avg("val"), F.sum("score")).count()
    elapsed = time.time() - t0
    print(f"  {n_parts:<14} {elapsed:>7.2f}s  {notes}")

In [ ]:
# ── Physical partitioning (partitionBy on write) ─────────────────────────────
# Partition the orders data by category — enables partition pruning on reads

parquet_parts = os.path.join(TMP, "orders_by_category")
df_bench_write = (spark.range(500_000)
    .withColumn("category", F.element_at(
        F.array(*[F.lit(c) for c in ["Electronics","Clothing","Food","Sports","Books"]]),
        (F.col("id") % 5 + 1).cast(IntegerType())))
    .withColumn("revenue", F.rand(seed=7)*500))

df_bench_write.write.mode("overwrite").partitionBy("category").parquet(parquet_parts)

print("Written partitioned Parquet. Directory layout:")
for entry in sorted(os.listdir(parquet_parts)):
    n_files = len(glob.glob(f"{parquet_parts}/{entry}/*.parquet"))
    print(f"  {entry}/  ({n_files} part file(s))")

# Partition pruning: reading one category only touches that subdirectory
t0 = time.time()
all_count = spark.read.parquet(parquet_parts).count()
t_all = time.time() - t0

t0 = time.time()
elec_count = spark.read.parquet(parquet_parts).filter(F.col("category")=="Electronics").count()
t_elec = time.time() - t0

print(f"\nRead all categories  ({all_count:>7,} rows): {t_all:.3f}s")
print(f"Read Electronics only ({elec_count:>7,} rows): {t_elec:.3f}s")
print("(Spark reads only the Electronics= subdirectory — partition pruning)")

### 5.4 Broadcast Joins

When joining a **large** table to a **small** one, the shuffle can be entirely eliminated
by broadcasting the small table to every executor.

**Sort-merge join (default for large + large):**
```
Large table  ─┐                  ┌─ Large table
               ├── SHUFFLE ──────┤
Small table  ─┘  (network I/O)   └─ Small table
                                          ↓ sort + merge per partition
```

**Broadcast hash join (large + small):**
```
Small table ─▶ Driver ─▶ broadcast ─▶ all executors
Large table ─────────────────────▶ hash join locally (no shuffle)
```

**When Spark auto-broadcasts:**
`spark.sql.autoBroadcastJoinThreshold` (default: 10 MB).
If the small table's estimated size is below this threshold, Spark automatically
chooses the broadcast strategy. You can also force it with `broadcast(df)`.

**When NOT to broadcast:**
- The "small" table exceeds driver memory → OOM on the driver
- Multiple concurrent jobs broadcast the same large table → memory pressure on executors
- The table is not actually small (Spark's size estimate can be wrong for non-Parquet sources)

In [ ]:
# ── Broadcast join benchmark ─────────────────────────────────────────────────
# Large fact table: 2M orders
df_fact = (spark.range(2_000_000)
    .withColumn("product_id", (F.col("id") % 200).cast(IntegerType()))
    .withColumn("revenue",    F.rand(seed=42)*500))

# Small dimension table: 200 products
df_dim = spark.createDataFrame(
    [(i, f"Product_{i}", ["Electronics","Clothing","Food","Sports","Books"][i%5],
      round(i*1.5+10,2)) for i in range(200)],
    ["product_id","product_name","category","list_price"])

# Disable auto-broadcast to force comparison
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

# Sort-merge join (no broadcast)
t0 = time.time()
smj_result = df_fact.join(df_dim, "product_id", "left")
smj_count  = smj_result.count()
t_smj = time.time() - t0

# Broadcast hash join
t0 = time.time()
bhj_result = df_fact.join(broadcast(df_dim), "product_id", "left")
bhj_count  = bhj_result.count()
t_bhj = time.time() - t0

# Restore
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", 10*1024*1024)

print(f"Sort-Merge Join   : {t_smj:.2f}s  ({smj_count:,} rows)")
print(f"Broadcast HashJoin: {t_bhj:.2f}s  ({bhj_count:,} rows)")
print(f"Speedup           : {t_smj/t_bhj:.1f}×")

# Verify plan shows BroadcastHashJoin
print("\n── Broadcast Join Physical Plan ──")
df_fact.join(broadcast(df_dim), "product_id", "left").explain()

### 5.5 Data Skew

**Data skew** occurs when one or a few partition keys contain a disproportionate fraction
of the data. The task processing the "hot" key takes far longer than all others,
becoming a **straggler** that blocks the entire stage.

```
Balanced:             Skewed:
Partition 0: ████     Partition 0: ████████████████████████████████  ← straggler
Partition 1: ████     Partition 1: ██
Partition 2: ████     Partition 2: █
Partition 3: ████     Partition 3: ██
Stage completes in    Stage completes when Partition 0 finishes
~4 time units         ~16 time units — 4× slower!
```

**Detection:** In the Spark UI, look for tasks with dramatically different durations.
Programmatically: check partition row counts after a groupBy on the suspect key.

**Fix — Salting:** Append a random integer (0..N-1) to the hot key, hash-join across N copies.
```python
# Add salt to hot key (e.g. customer_id = 1 is hot)
df_fact.withColumn("salt", (F.rand()*N).cast(IntegerType()))
       .withColumn("salted_key", F.concat("customer_id","salt"))

# Replicate the lookup table N times
df_lookup_salted = df_lookup.crossJoin(spark.range(N).withColumn("salt",F.col("id")))
                             .withColumn("salted_key", F.concat("customer_id","salt"))
```

In [ ]:
# ── Skew detection ───────────────────────────────────────────────────────────
import random

# Create artificially skewed dataset: customer 1 has 80% of all orders
n_total = 300_000
n_hot   = int(n_total * 0.80)
n_rest  = n_total - n_hot

skewed_rows = (
    [(1, round(random.uniform(5,500),2)) for _ in range(n_hot)] +
    [(random.randint(2,1000), round(random.uniform(5,500),2)) for _ in range(n_rest)]
)
df_skewed = spark.createDataFrame(skewed_rows, ["customer_id","revenue"])
df_skewed = df_skewed.repartition(8)

print("Partition row counts (shows skew):")
(df_skewed
 .groupBy(F.spark_partition_id().alias("partition"))
 .count()
 .orderBy(F.desc("count"))
 .show())

print("Customer distribution (top 5):")
(df_skewed
 .groupBy("customer_id")
 .count()
 .orderBy(F.desc("count"))
 .show(5))

### 5.6 Adaptive Query Execution (AQE)

Introduced in Spark 3.0, **Adaptive Query Execution** improves performance by making
runtime decisions based on **actual data statistics** collected during execution —
rather than relying solely on pre-execution estimates.

Three key optimizations AQE provides automatically:

| Optimization | Problem solved | How |
|-------------|---------------|-----|
| **Coalescing shuffle partitions** | 200 empty/tiny partitions after a filter | Merges small post-shuffle partitions at runtime |
| **Switching join strategies** | Catalyst chose sort-merge join, but one side is actually small | Switches to broadcast join after seeing actual sizes |
| **Skew join optimization** | One partition is 10× larger than others | Splits skewed partitions and processes them in parallel |

**Enable AQE (Spark 3.x):**
```python
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
```

AQE is enabled by default in Spark 3.2+. It does not require code changes.

In [ ]:
# ── AQE: Automatic partition coalescing ───────────────────────────────────────
# Set high shuffle partitions, then filter heavily — AQE merges the empties

spark.conf.set("spark.sql.shuffle.partitions", "200")
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")

df_aqe = make_df(1_000_000).filter(F.col("cat") == "0")  # ~5% of data

t0 = time.time()
result = df_aqe.groupBy("cat").agg(F.avg("val")).count()
t_aqe_on = time.time() - t0

# Disable AQE and compare
spark.conf.set("spark.sql.adaptive.enabled", "false")
t0 = time.time()
result = df_aqe.groupBy("cat").agg(F.avg("val")).count()
t_aqe_off = time.time() - t0

# Restore
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.shuffle.partitions", "8")

print(f"AQE ON  (coalesces empty partitions): {t_aqe_on:.2f}s")
print(f"AQE OFF (200 mostly-empty partitions): {t_aqe_off:.2f}s")
print(f"AQE speedup: {t_aqe_off/t_aqe_on:.1f}×")
print("\n(With 200 shuffle partitions and only ~5% data passing the filter,")
print(" AQE merges the ~190 near-empty partitions into a few meaningful ones)")

### 5.7 Reading the Spark UI and Query Plans

The **Spark UI** (accessible at `spark.sparkContext.uiWebUrl`) is your primary debugging tool.

| Tab | What to look for |
|-----|-----------------|
| **Jobs** | Total job duration; failed jobs |
| **Stages** | Stage duration; task distribution (skew = some tasks 10× slower) |
| **SQL/DataFrame** | Visual query plan; node timings; rows per node |
| **Storage** | Cached DataFrames; memory used |
| **Executors** | GC time (>10% = memory pressure); task failures |

**Reading `explain()` output (bottom-up):**
```
== Physical Plan ==
*(2) HashAggregate(...)          ← final aggregation (stage 2)
+- Exchange hashpartitioning(...)← SHUFFLE (stage boundary)
   +- *(1) HashAggregate(...)    ← partial aggregation (stage 1)
      +- *(1) Filter (...)       ← filter pushed down
         +- *(1) ColumnarToRow   ← reading Parquet
            +- Scan parquet ...  ← data source (bottom = start)
```

Asterisks `*(n)` indicate **whole-stage code generation** — Tungsten has compiled this node
into a single bytecode loop. Nodes without asterisks are interpreted — potential optimization targets.

In [ ]:
# ── Read and interpret query plans ──────────────────────────────────────────
parquet_path = os.path.join(TMP.replace("/tmp/spark_module5",""), "employees") if os.path.exists(
    "/tmp/employees") else os.path.join(TMP, "employees_perf")

# Write a fresh dataset for this section
(spark.range(300_000)
 .withColumn("dept",   F.element_at(F.array(*[F.lit(d) for d in ["Eng","Mkt","Fin","HR"]]),
                                     (F.col("id")%4+1).cast(IntegerType())))
 .withColumn("salary", F.rand(seed=1)*80000+30000)
 .withColumn("age",    (F.rand(seed=2)*40+22).cast(IntegerType()))
 .write.mode("overwrite").parquet(parquet_path))

df_p = spark.read.parquet(parquet_path)

complex_query = (
    df_p
    .filter(F.col("salary") > 50000)
    .filter(F.col("age").between(25, 55))
    .groupBy("dept")
    .agg(F.avg("salary").alias("avg_sal"),
         F.count("*").alias("headcount"))
    .filter(F.col("headcount") > 10)
    .orderBy(F.desc("avg_sal"))
)

print("=== Simple mode (physical plan only) ===")
complex_query.explain()

print("\n=== Cost mode (with row count estimates) ===")
complex_query.explain("cost")

print(f"\nSpark UI: {spark.sparkContext.uiWebUrl}")
print("→ Navigate to the SQL tab to see a visual DAG after running complex_query.show()")
complex_query.show()

### Section 5 — Exercises

**Exercise 5.1 — Shuffle Minimization**

You are given the following pipeline:

```python
df = spark.read.parquet("orders.parquet")  # 50M rows, 8 columns

result = (df
    .withColumn("revenue_tier",
        F.when(F.col("revenue")>1000,"High").otherwise("Low"))
    .groupBy("category","revenue_tier")
    .agg(F.sum("revenue").alias("total"))
    .join(category_lookup, "category", "left")
    .filter(F.col("total") > 100000)
    .orderBy(F.desc("total")))
```

1. Count the number of shuffle operations (stage boundaries) in this pipeline. Justify each one.
2. `category_lookup` has 20 rows. Rewrite the join to eliminate its shuffle. Show the code and verify with `.explain()`.
3. The `filter(total > 100000)` appears after the join. Can it be moved earlier? Why or why not? What would Catalyst do?
4. Propose a rewrite of the full pipeline that minimizes shuffles. Benchmark the original vs your rewrite on the `make_orders(2_000_000)` DataFrame.

---

**Exercise 5.2 — Caching Strategy**

A data pipeline runs the following operations on the same base DataFrame:

```
df_base  →  agg_by_city      (groupBy city — used in report A and report B)
df_base  →  agg_by_category  (groupBy category — used in report C only)
df_base  →  high_value_filter (filter revenue>500 — used in reports B and D)
```

1. Draw the dependency graph. Identify which DataFrames should be cached and which should not. Justify each decision using the three caching criteria.
2. Implement the pipeline with correct caching. Measure total wall-clock time with and without caching.
3. A colleague suggests: *"Just cache `df_base` — then everything downstream is fast."* Evaluate this claim: when is it correct, and when does it hurt performance?

---

**Exercise 5.3 — Partition Tuning**

1. Generate `make_orders(5_000_000)` and measure groupBy performance for shuffle partition counts of: 2, 4, 8, 16, 32, 100.
2. Plot (or tabulate) time vs partition count. Identify the optimal value for your machine.
3. Explain in terms of task scheduling overhead and core utilization why both extremes are slow.
4. A production cluster has 20 nodes × 8 cores = 160 cores total.
   What value of `spark.sql.shuffle.partitions` would you recommend? Show your reasoning.
5. The same cluster processes datasets ranging from 1 GB to 1 TB.
   Should you set `shuffle.partitions` dynamically per job, or use AQE? Explain.

In [ ]:
# ── Exercise 5.1 workspace ───────────────────────────────────────────────────
category_lookup = spark.createDataFrame(
    [("Electronics","CE"),("Clothing","FAS"),("Food","FMCG"),("Sports","SPT"),("Books","EDU")],
    ["category","sector"])

df_orders_ex = make_orders(2_000_000)

# Original pipeline (count its shuffles using explain())
original = (
    df_orders_ex
    .withColumn("revenue_tier", F.when(F.col("revenue")>300,"High").otherwise("Low"))
    .groupBy("category","revenue_tier").agg(F.sum("revenue").alias("total"))
    .join(category_lookup,"category","left")
    .filter(F.col("total")>50000)
    .orderBy(F.desc("total"))
)
print("Original plan:")
original.explain()

# TODO: rewrite with broadcast join
# TODO: benchmark original vs rewrite

# ── Exercise 5.2 workspace ───────────────────────────────────────────────────
# TODO: implement the 4-report pipeline with correct caching

# ── Exercise 5.3 workspace ───────────────────────────────────────────────────
df_5m = make_orders(5_000_000)
print("\nPartition count vs groupBy time:")
print(f"  {'shuffle.partitions':<22} {'Time (s)':>10}")
print("  " + "─"*35)
for n in [2, 4, 8, 16, 32, 100]:
    spark.conf.set("spark.sql.shuffle.partitions", str(n))
    t0 = time.time()
    df_5m.groupBy("category","product_id").agg(F.sum("revenue")).count()
    elapsed = time.time() - t0
    print(f"  {n:<22} {elapsed:>10.2f}")
spark.conf.set("spark.sql.shuffle.partitions", "8")

---
<h2 style="color:#E25822;font-family:Arial;">Module Summary</h2>

### Core Concepts Reference

| Topic | Key Idea | Practical Implication |
|-------|----------|-----------------------|
| **Driver/Executor** | Driver orchestrates; Executors compute | Increase driver memory for `collect()` and broadcasts |
| **Lazy evaluation** | Transformations are recorded; actions trigger execution | Chain transformations freely; only actions cost |
| **Narrow vs Wide** | Narrow: no shuffle; Wide: shuffle required | Minimize groupBy/join; filter early |
| **Catalyst** | 4-phase optimizer: Analysis → Logical → Physical → Codegen | Use built-in F.* functions; write filters first |
| **Tungsten** | Off-heap memory, compact binary, whole-stage codegen | Avoid Python UDFs; use pandas_udf if unavoidable |
| **Fault tolerance** | Lineage-based recovery; checkpointing for long lineages | Use checkpointing for iterative algorithms |
| **Watermarking** | Bounds streaming state; drops events older than threshold | Set Δ ≥ 2× observed late-arrival latency |
| **ML Pipelines** | Chains Estimators and Transformers; prevents data leakage | Always split data before calling `pipeline.fit()` |
| **Caching** | Avoids recomputation for reused DataFrames | Cache if: expensive + reused + fits in memory |
| **Shuffle partitions** | Controls post-shuffle parallelism | Set to 2–4× cores; use AQE for variable workloads |
| **Broadcast join** | Eliminates shuffle for small dimension tables | Auto-broadcast < 10 MB; force with `broadcast(df)` |
| **AQE** | Runtime adaptation: coalesce partitions, switch join, fix skew | Enable always in Spark 3.x; no code changes needed |

### Further Reading

| Resource | Scope |
|----------|-------|
| [Spark Documentation](https://spark.apache.org/docs/latest/) | Official reference |
| [Learning Spark 2nd Ed. (O'Reilly)](https://www.oreilly.com/library/view/learning-spark-2nd/9781492050032/) | Chapters 4–8, 12 |
| [Spark: The Definitive Guide (O'Reilly)](https://www.oreilly.com/library/view/spark-the-definitive/9781491912201/) | Deep internals |
| [Databricks Engineering Blog](https://www.databricks.com/blog/category/engineering) | AQE, Photon, Delta Lake |
| [Catalyst Paper (SIGMOD 2015)](https://dl.acm.org/doi/10.1145/2723372.2742797) | Original Catalyst research |
| [Tungsten Deep Dive (Databricks)](https://www.databricks.com/blog/2015/04/28/project-tungsten-bringing-spark-closer-to-bare-metal.html) | Tungsten internals |

---
*Module 5 — Apache Spark: Theory and Practice &nbsp;|&nbsp; Big Data Analytics — Undergraduate Course*

In [ ]:
# ── Cleanup ───────────────────────────────────────────────────────────────────
import shutil

for q in spark.streams.active:
    q.stop()
    print(f"Stopped streaming query: {q.id}")

for d in ["/tmp/spark_module5", "/tmp/chk_s3_rate", "/tmp/chk_s3_file",
          "/tmp/chk_s3_win", "/tmp/chk_s3_wm", "/tmp/chk_s3_etl"]:
    if os.path.exists(d):
        shutil.rmtree(d, ignore_errors=True)

print("Cleanup complete.")
print(f"SparkSession still active: {spark.version}")
print("Call spark.stop() when you are completely done with this session.")